# Pan-UK Biobank: a multi-million-variant Manhattan plot

Each Pan-UKBB per-phenotype file contains 28,987,534 variants. This
notebook reads indexed windows from the standing-height GWAS and plots
the released `−log10(p)` statistic across all autosomes. The full
2.0 GB flat file stays remote; only its 2 MB tabix index and requested
BGZF blocks are transferred.

Increase `PANUKBB_WINDOWS` or `PANUKBB_VARIANTS_PER_WINDOW` for a
denser run.

**Source:** [Pan-UKBB downloads](https://pan.ukbb.broadinstitute.org/downloads/index.html)
and [per-phenotype file documentation](https://pan.ukbb.broadinstitute.org/docs/per-phenotype-files/index.html).
The data are CC BY 4.0; publications should acknowledge Pan-UKBB and
UK Biobank as requested on the download page.

Install beside XY with `python -m pip install numpy pysam requests xy`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pysam
import requests

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/"
    "continuous-50-both_sexes-irnt.tsv.bgz"
)
INDEX_URL = (
    "https://pan-ukb-us-east-1.s3.amazonaws.com/"
    "sumstats_flat_files_tabix/"
    "continuous-50-both_sexes-irnt.tsv.bgz.tbi"
)
index_path = DATA_DIR / "continuous-50-both_sexes-irnt.tsv.bgz.tbi"
if not index_path.exists():
    response = requests.get(INDEX_URL, timeout=120)
    response.raise_for_status()
    index_path.write_bytes(response.content)

CHROMOSOME_LENGTHS = {
    1: 249_250_621,
    2: 243_199_373,
    3: 198_022_430,
    4: 191_154_276,
    5: 180_915_260,
    6: 171_115_067,
    7: 159_138_663,
    8: 146_364_022,
    9: 141_213_431,
    10: 135_534_747,
    11: 135_006_516,
    12: 133_851_895,
    13: 115_169_878,
    14: 107_349_540,
    15: 102_531_392,
    16: 90_354_753,
    17: 81_195_210,
    18: 78_077_248,
    19: 59_128_983,
    20: 63_025_520,
    21: 48_129_895,
    22: 51_304_566,
}
windows_per_chromosome = int(os.getenv("PANUKBB_WINDOWS", "8"))
variants_per_window = int(os.getenv("PANUKBB_VARIANTS_PER_WINDOW", "15000"))
window_width = int(os.getenv("PANUKBB_WINDOW_BP", "3000000"))
if min(windows_per_chromosome, variants_per_window, window_width) <= 0:
    raise ValueError("Pan-UKBB window controls must be positive")

In [ ]:
offsets = {}
running_offset = 0
for chromosome, length in CHROMOSOME_LENGTHS.items():
    offsets[chromosome] = running_offset
    running_offset += length

position_parts = []
significance_parts = []
chromosome_parts = []

# The immutable quantitative-trait schema starts with:
# chr, pos, ref, alt, af_meta_hq, beta_meta_hq, se_meta_hq,
# neglog10_pval_meta_hq. Its plain TSV header is skipped by tabix.
position_column = 1
pvalue_column = 7

with pysam.TabixFile(DATA_URL, index=str(index_path)) as summary:
    for chromosome, length in CHROMOSOME_LENGTHS.items():
        centers = (
            (np.arange(windows_per_chromosome, dtype=np.float64) + 0.5)
            * length
            / windows_per_chromosome
        )
        starts = np.clip(
            centers - window_width / 2,
            0,
            max(0, length - window_width),
        ).astype(np.int64)
        positions = []
        significance = []
        for start in starts:
            kept = 0
            records = summary.fetch(
                str(chromosome),
                int(start),
                int(start + window_width),
            )
            for line in records:
                fields = line.split("\t")
                value = fields[pvalue_column]
                if value == "NA":
                    continue
                positions.append(offsets[chromosome] + int(fields[position_column]))
                significance.append(float(value))
                kept += 1
                if kept >= variants_per_window:
                    break

        position_parts.append(np.asarray(positions, dtype=np.float64))
        significance_parts.append(np.asarray(significance, dtype=np.float64))
        chromosome_parts.append(np.full(len(positions), chromosome, dtype=np.float64))
        print(f"chr{chromosome}: {len(positions):,} variants")

genomic_position = np.concatenate(position_parts)
neglog10_pvalue = np.concatenate(significance_parts)
chromosome_number = np.concatenate(chromosome_parts)
print(f"{genomic_position.size:,} variants total")

In [ ]:
tick_values = [
    offsets[chromosome] + CHROMOSOME_LENGTHS[chromosome] / 2 for chromosome in CHROMOSOME_LENGTHS
]
tick_labels = [str(chromosome) for chromosome in CHROMOSOME_LENGTHS]
chromosome_boundaries = [offsets[chromosome] for chromosome in range(2, 23)]

# Two fixed positions in the plasma palette alternate indigo and ochre.
# This separates adjacent chromosomes without implying a continuous scale.
alternating_color = np.where(chromosome_number % 2, 0.18, 0.78)

chart = xy.scatter_chart(
    xy.scatter(
        genomic_position,
        neglog10_pvalue,
        color=alternating_color,
        color_domain=(0, 1),
        colormap="plasma",
        size=1.35,
        opacity=0.78,
        density=True,
    ),
    *[
        xy.vline(
            boundary,
            color="#b7ab9b",
            width=0.7,
            opacity=0.46,
            style={"dash": "2,5"},
        )
        for boundary in chromosome_boundaries
    ],
    xy.hline(
        -np.log10(5e-8),
        text="Genome-wide threshold · P = 5 \u00d7 10⁻⁸",
        color="#a33a32",
        width=2.3,
        style={"dash": "7,4"},
    ),
    xy.text(
        0.071,
        0.965,
        "PAN-UK BIOBANK  /  STANDING-HEIGHT GWAS",
        dx=0,
        dy=0,
        color="#50354a",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 21,
            "font_weight": 750,
            "letter_spacing": "0.025em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.071,
        0.918,
        f"{genomic_position.size:,} VARIANTS  ·  22 AUTOSOMES  ·  PAN-ANCESTRY META-ANALYSIS",
        dx=0,
        dy=0,
        color="#766b60",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10.5,
            "font_weight": 600,
            "letter_spacing": "0.08em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.929,
        0.952,
        "GENOME-WIDE THRESHOLD  /  P = 5 \u00d7 10⁻⁸",
        dx=0,
        dy=0,
        color="#a33a32",
        anchor="end",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 10.5,
            "font_weight": 750,
            "letter_spacing": "0.055em",
            "vertical_align": "top",
        },
    ),
    xy.text(
        0.52,
        0.04,
        "CHROMOSOME",
        dx=0,
        dy=0,
        color="#342f2a",
        anchor="middle",
        style={
            "coordinate_space": "figure_fraction",
            "font_size": 11,
            "font_weight": 700,
            "letter_spacing": "0.07em",
        },
    ),
    xy.x_axis(
        label=None,
        tick_values=tick_values,
        tick_labels=tick_labels,
        style={
            "grid_opacity": 0,
            "axis_color": "#6f665c",
            "axis_width": 1.2,
            "tick_color": "#8f8375",
            "tick_width": 1,
            "tick_length": 5,
            "tick_label_color": "#4a433c",
            "label_color": "#342f2a",
        },
    ),
    xy.y_axis(
        label="Association significance, \u2212log₁₀(P)",
        label_offset=-16,
        tick_count=7,
        style={
            "grid_color": "#d8d0c3",
            "grid_width": 1,
            "grid_dash": "dotted",
            "grid_opacity": 0.9,
            "axis_color": "#6f665c",
            "axis_width": 1.2,
            "tick_color": "#8f8375",
            "tick_label_color": "#4a433c",
            "label_color": "#342f2a",
        },
    ),
    xy.legend(show=False),
    xy.tooltip(
        title="Standing-height association",
        format={"x": ",.0f", "y": ".2f"},
    ),
    xy.interaction_config(
        crosshair=True,
        wheel_zoom=True,
        box_zoom=True,
        double_click_reset=True,
    ),
    xy.theme(
        background="#f3eee4",
        plot_background="#fffdf8",
        text_color="#2c2824",
        grid_color="#d8d0c3",
        axis_color="#6f665c",
        crosshair_color="#a33a32",
        selection_color="#71465f",
        selection_fill="#71465f24",
    ),
    styles={
        "axis_title": {"font_weight": 650, "letter_spacing": "0.035em"},
        "tick_label": {"font_variant_numeric": "tabular-nums"},
        "annotation_label": {"font_weight": 650},
    },
    style={
        "border": "1px solid #d4cabc",
        "font_family": "Georgia, 'Times New Roman', serif",
    },
    width=1150,
    height=620,
    padding=(98, 38, 94, 112),
)
print(chart.memory_report()["canonical_bytes"], "canonical bytes")
chart